# Benchmark Registry Overview

This notebook creates a compact registry view of ChaosWing's exported experiment rows so forecasting, ranking, trust, and lead-lag work can be scanned from one table.

In [ ]:
from pathlib import Path
import json

import pandas as pd

DATA_DIR = Path("../ml_data") if Path("../ml_data").exists() else Path("ml_data")

def load_jsonl(path: Path) -> pd.DataFrame:
    if not path.exists():
        return pd.DataFrame()
    with path.open("r", encoding="utf-8") as handle:
        rows = [json.loads(line) for line in handle if line.strip()]
    return pd.DataFrame(rows)

experiments_df = load_jsonl(DATA_DIR / "experiments.jsonl")
experiments_df.head()

In [ ]:
PRIMARY_METRIC_BY_TASK = {
    "resolution_backtest": "model_brier",
    "related_market_ranking": "model_ndcg_at_5",
    "related_market_usefulness": "model_ndcg_at_5",
    "agent_trust": "avg_trust_score",
    "quality_backtest": "rmse",
    "leadlag_backtest": "net_pnl",
}

BASELINE_BY_TASK = {
    "resolution_backtest": "market-implied YES probability",
    "related_market_ranking": "lexical overlap",
    "related_market_usefulness": "reviewer-aware labels",
    "agent_trust": "structural support checks",
    "quality_backtest": "heuristic scorer",
    "leadlag_backtest": "screened candidate signals",
}

records = []
for _, row in experiments_df.iterrows():
    task_type = row.get("task_type")
    metrics = row.get("metrics") or {}
    primary_key = PRIMARY_METRIC_BY_TASK.get(task_type)
    records.append({
        "task_type": task_type,
        "title": row.get("title"),
        "dataset_version": row.get("dataset_version"),
        "baseline": BASELINE_BY_TASK.get(task_type, "n/a"),
        "primary_metric": primary_key,
        "primary_value": metrics.get(primary_key) if primary_key else None,
        "readiness": "live benchmark" if task_type in {"resolution_backtest", "related_market_ranking", "related_market_usefulness", "agent_trust"} else "research track",
    })

registry_df = pd.DataFrame(records)
registry_df.sort_values(["task_type", "title"], na_position="last")


## Findings

- Summarize which tasks have stable, repeatable benchmark outputs today.
- Note where the registry already shows baseline-vs-challenger comparisons clearly.

## Limitations

- Record any tasks that still surface sparse experiment history.
- Note where richer metadata or timestamp normalization would improve the registry.